# Codeforces RAG Chatbot Example

This notebook demonstrates the Assignment 2 pipeline: CodeBERT embeddings, FAISS retrieval, grounded answer generation, and conversation history. Run `vector_store.py` once before these cells so `data/index` exists.

In [ ]:
import os

# Needed only on macOS environments with a PyTorch/FAISS OpenMP conflict.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

from embeddings import CodeBERTEmbedder

embedder = CodeBERTEmbedder(device="cpu")
single = embedder.generate_embedding("Find the maximum subarray sum")
batch = embedder.batch_generate_embeddings([
    "Implement a segment tree",
    "Solve the knapsack problem",
])
print("Single embedding shape:", single.shape)
print("Batch embedding shape:", batch.shape)

In [ ]:
from pathlib import Path
from retriever import ProblemRetriever

retriever = ProblemRetriever(Path("data/index"), device="cpu")
results = retriever.retrieve(
    "Explain the main idea for solving Even Simple Path",
    top_k=3,
)
[(item["problem_id"], item["title"], item["section"]) for item in results]

In [ ]:
import gc

del embedder, retriever
gc.collect()

from chatbot import CodeforcesChatbot

chatbot = CodeforcesChatbot(Path("data/index"), device="cpu", top_k=3)
answer, sources = chatbot.ask(
    "Explain the main idea for solving Even Simple Path."
)
print(answer)
print([(item["problem_id"], item["section"]) for item in sources])

In [ ]:
follow_up, follow_up_sources = chatbot.ask(
    "How is that matching constructed?"
)
print(follow_up)
print("History messages:", len(chatbot.history))
print([item["problem_id"] for item in follow_up_sources])